# IndabaX Zimbabwe 2026 — Loan Default Prediction

**Deep Learning IndabaX Zimbabwe 2026** | AI for Financial Inclusion

This notebook runs the **complete pipeline** end-to-end:
1. Clone the repo and install dependencies
2. Load competition data (Drive mount or upload)
3. Feature engineering (3 encoding variants)
4. Train models (LGBM, XGBoost, CatBoost, TabNet, MLP)
5. Ensemble and submit

**Runtime:** Set to **GPU (T4)** for best results. Expected ~30 min.

## 1. Environment Setup

In [ ]:
import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
except Exception:
    gpu_name = 'CPU'
print(f'Device: {gpu_name}')

## 2. Clone Repo & Install

In [ ]:
REPO_URL = 'https://github.com/TinevimboMusingadi/indabax_zimbabwe_hackathon.git'
REPO_DIR = 'indabax_zimbabwe_hackathon'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
    !pip install -q -e .
else:
    print('Local mode — ensure you have run: pip install -e .[dev]')

## 3. Load Competition Data

**Option A (recommended):** Upload `Train.csv`, `Test.csv`, `SampleSubmission.csv` to
`Google Drive > My Drive > indabax_data/`, then set `DATA_SOURCE = 'drive'`.

**Option B:** Set `DATA_SOURCE = 'upload'` and upload files via the dialog.

In [ ]:
DATA_SOURCE = 'drive'  # 'drive' or 'upload'

if IN_COLAB:
    os.makedirs('data/raw', exist_ok=True)

    if DATA_SOURCE == 'drive':
        from src.utils.colab import mount_drive
        mount_drive()
    elif DATA_SOURCE == 'upload':
        from src.utils.colab import upload_files
        upload_files()
    else:
        raise ValueError(f'Unknown DATA_SOURCE: {DATA_SOURCE}')
else:
    print('Local mode — ensure CSVs are in data/raw/')

## 4. Verify Data

In [ ]:
import os
for f in ['Train.csv', 'Test.csv', 'SampleSubmission.csv']:
    path = os.path.join('data', 'raw', f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'{f}: {size_mb:.1f} MB')
    else:
        print(f'MISSING: {path}')

## 5. Run Full Pipeline

This executes the entire pipeline:
- Data loading + validation
- Stratified CV fold creation
- Feature engineering (all variants)
- Model training with OOF predictions
- Optional Optuna tuning
- Ensemble (rank avg + stacking + blend)
- Submission CSV generation

In [ ]:
# Choose config based on your runtime:
#   colab_t4.yaml  — GPU T4, ~30 min, full model suite
#   fast.yaml      — CPU, ~5 min, LGBM + XGB only

CONFIG = 'configs/colab_t4.yaml' if IN_COLAB else 'configs/fast.yaml'
print(f'Using config: {CONFIG}')

!python -m src --config {CONFIG}

## 6. Review Results

In [ ]:
import pandas as pd
import glob

# Show leaderboard
lb_path = 'results/leaderboard.md'
if os.path.exists(lb_path):
    print(open(lb_path).read())

# Show latest submission
sub_files = sorted(glob.glob('submissions/sub_*.csv'))
if sub_files:
    latest = sub_files[-1]
    print(f'\nLatest submission: {latest}')
    sub = pd.read_csv(latest)
    print(f'Shape: {sub.shape}')
    print(f'Columns: {list(sub.columns)}')
    print(f'Target stats: min={sub.Target.min():.4f}, max={sub.Target.max():.4f}, mean={sub.Target.mean():.4f}')
    print(sub.head(10))
else:
    print('No submission files found.')

## 7. Download Submission

In [ ]:
if IN_COLAB and sub_files:
    from google.colab import files
    files.download(sub_files[-1])
    print(f'Downloaded: {sub_files[-1]}')
elif sub_files:
    print(f'Submission ready at: {sub_files[-1]}')
else:
    print('No submission to download.')